# 02 — Vocal2BGM: gerar arranjo a partir da sua voz

Este notebook usa o **ACE-Step 1.5** para gerar um arranjo musical completo  
ao redor da sua voz isolada. Internamente, mapeia para `task_type="complete"`,  
que pede ao modelo para **preencher os instrumentos faltantes** a partir do vocal  
("Fill in missing tracks: just vocals → drums, bass, guitar, etc.").

**Entrada:** `audio/stems/htdemucs/<nome>/vocals.wav` (do notebook 01)  
**Saída:** arquivo gerado em `audio/output/`

---
### Como funciona aqui

Em vez de chamar a API Python diretamente (que exige inicializar `AceStepHandler`,  
`LLMHandler`, configurar GPU, etc.), montamos um arquivo TOML com os parâmetros e  
invocamos a CLI oficial do ACE-Step (`cli.py -c config.toml`). Mais simples, mais  
estável, e o mesmo TOML pode ser rodado fora do notebook se quiser.

### Parâmetros principais

| Parâmetro | Faixa | Efeito |
|---|---|---|
| `inference_steps` | 8–50 | mais passos = melhor qualidade (mais lento) |
| `guidance_scale` | 4–10 | adesão ao caption: maior = mais fiel |
| `complete_tracks` | csv | quais instrumentos a IA deve gerar |
| `duration` | seg | duração-alvo (use perto da duração do vocal) |

### Atenção para 4 GB VRAM (GTX 1050 Ti)

- `task_type="complete"` exige o **modelo base** (não turbo) — mais pesado.
- Use `offload_to_cpu=true` no TOML (geração ficará lenta mas evita OOM).
- Comece com `inference_steps=8` e `duration` baixa (~20s) para validar.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from scripts.utils import (
    carregar_audio, plotar_audio, log_sessao, verificar_gpu,
    AUDIO_STEMS, AUDIO_OUT
)

ACE_STEP_ROOT = Path('..').resolve() / 'ACE-Step-1.5'
assert ACE_STEP_ROOT.exists(), f'ACE-Step não encontrado em {ACE_STEP_ROOT}. Execute o notebook 00 primeiro.'

device = verificar_gpu()

In [ ]:
# ── CONFIGURAÇÃO — edite aqui ─────────────────────────────────────────────────

# Caminho da voz isolada (saída do notebook 01)
VOCAL_PATH = AUDIO_STEMS / 'htdemucs' / 'minha_musica' / 'vocals.wav'

# Descrição do arranjo desejado (em inglês funciona melhor — modelo treinado predominantemente em EN)
CAPTION = (
    'brazilian samba, acoustic guitar, cavaquinho, pandeiro, surdo, '
    'acoustic bass, male vocals, joyful, moderate tempo, clean production'
)

# Quais tracks o modelo deve GERAR (sua voz fica intacta — não inclua "vocals" aqui)
# Opções: drums, bass, guitar, keyboard, percussion, strings, synth, fx, brass, woodwinds, backing_vocals
COMPLETE_TRACKS = 'drums,bass,guitar,percussion'

# Parâmetros de geração
INFERENCE_STEPS = 8        # 8 = rápido (validar); 32–50 = qualidade final
GUIDANCE_SCALE  = 7.0      # 4.0–10.0
DURATION_SEC    = 20.0     # comece curto para iterar
SEED            = -1       # -1 = aleatório

# Nome do arquivo de saída (sem extensão)
NOME_SAIDA = 'samba_01_bgm'

# ─────────────────────────────────────────────────────────────────────────────

assert VOCAL_PATH.exists(), f'Voz não encontrada: {VOCAL_PATH}\nExecute primeiro o notebook 01.'
vocal_audio, sr_vocal = carregar_audio(VOCAL_PATH)
plotar_audio(vocal_audio, sr_vocal, titulo='Voz isolada (entrada)')

In [ ]:
# ── Gerar arquivo TOML de configuração para o ACE-Step CLI ────────────────────

import toml
import tempfile

config = {
    # Modelo
    'project_root': str(ACE_STEP_ROOT),
    'config_path': 'acestep-v15-base',   # COMPLETE exige modelo base, não turbo
    'device': 'auto',
    'offload_to_cpu': True,               # CRÍTICO em 4 GB VRAM
    'backend': 'pt',                      # backend do LM: 'pt' = PyTorch (vllm exige Linux)

    # Tarefa
    'task_type': 'complete',
    'src_audio': str(VOCAL_PATH),
    'complete_tracks': COMPLETE_TRACKS,
    'caption': CAPTION,
    'lyrics': '',                         # vazio — o modelo só vai gerar instrumentos
    'instrumental': False,                # mantém o vocal

    # Geração
    'inference_steps': INFERENCE_STEPS,
    'guidance_scale': GUIDANCE_SCALE,
    'duration': DURATION_SEC,
    'seed': SEED,
    'batch_size': 1,
    'audio_format': 'wav',

    # LM — desabilitar para economizar VRAM (deixa caption como está, sem CoT)
    'thinking': False,
    'use_cot_metas': False,
    'use_cot_caption': False,
    'use_cot_language': False,

    # Output
    'save_dir': str(AUDIO_OUT.resolve()),
}

config_path = Path(tempfile.gettempdir()) / f'ace_config_{NOME_SAIDA}.toml'
with open(config_path, 'w', encoding='utf-8') as f:
    toml.dump(config, f)

print(f'✓ Config TOML escrito em: {config_path}')
print('--- conteúdo ---')
print(config_path.read_text(encoding='utf-8'))

In [ ]:
# ── Executar ACE-Step CLI ─────────────────────────────────────────────────────
# Primeira execução baixa ~5 GB de modelos (cache em ~/.cache/huggingface)

import subprocess
import time

cli_script = ACE_STEP_ROOT / 'cli.py'
cmd = [sys.executable, str(cli_script), '-c', str(config_path)]

print(f'Executando: {" ".join(cmd)}\n')
t0 = time.time()

resultado = subprocess.run(cmd, capture_output=False)

elapsed = time.time() - t0
if resultado.returncode == 0:
    print(f'\n✓ Geração concluída em {elapsed:.1f}s')
else:
    print(f'\n✗ Erro (exit {resultado.returncode}) — veja log acima.')

In [ ]:
# ── Localizar saída mais recente e reproduzir ─────────────────────────────────

from IPython.display import Audio, display

saidas = sorted(AUDIO_OUT.glob('*.wav'), key=lambda p: p.stat().st_mtime, reverse=True)
assert saidas, f'Nenhum WAV encontrado em {AUDIO_OUT}. Verifique o log da CLI acima.'

saida_recente = saidas[0]
print(f'Arquivo mais recente: {saida_recente.name}  ({saida_recente.stat().st_size/1e6:.1f} MB)')

audio_out, sr_out = carregar_audio(saida_recente)
plotar_audio(audio_out, sr_out, titulo=f'Resultado: {saida_recente.name}')

print('\nEntrada (voz isolada):')
display(Audio(str(VOCAL_PATH)))

print('\nSaída (música completa):')
display(Audio(str(saida_recente)))

In [ ]:
# ── REGISTRO DA SESSÃO — edite as observações antes de executar ───────────────

log_sessao(
    titulo=f'Vocal2BGM — {NOME_SAIDA}',
    notas=f"""
- Voz de entrada: {VOCAL_PATH.name}
- Caption: {CAPTION}
- Tracks gerados: {COMPLETE_TRACKS}
- inference_steps: {INFERENCE_STEPS}
- guidance_scale: {GUIDANCE_SCALE}
- duration: {DURATION_SEC}s
- Tempo de geração: {elapsed:.1f}s
- Saída: {saida_recente.name}
- Avaliação: (1–5 estrelas)
- O que ficou bom:
- O que ficou ruim:
- Próximo teste: (ex: aumentar steps, mudar caption, trocar tracks)
"""
)

## Experimentos sugeridos

Mude os parâmetros e gere de novo. Cada experimento fica registrado em `docs/sessoes.md`.

| Teste | inference_steps | guidance | mudança |
|---|---|---|---|
| 01 | 8 | 7.0 | base — validar pipeline |
| 02 | 32 | 7.0 | mais qualidade |
| 03 | 32 | 9.0 | mais fiel ao caption |
| 04 | 32 | 5.0 | mais criativo |
| 05 | 32 | 7.0 | mudar `COMPLETE_TRACKS` (ex: só `drums,bass`) |
| 06 | 32 | 7.0 | mudar `CAPTION` para outro gênero |

## Troubleshooting

- **OOM (out of memory):** baixe `DURATION_SEC` e `INFERENCE_STEPS`. Confirme `offload_to_cpu=True` no TOML.
- **Geração muito lenta:** esperado em 4 GB VRAM. Para iterar, mantenha `inference_steps=8` e `duration=15`.
- **Download dos modelos (~5 GB) na primeira execução:** vai para `~/.cache/huggingface`. Cache persistente entre execuções.
- **"src_audio is required":** o vocal não existe no caminho informado — re-rode o notebook 01.
- **"requires a base model config":** o `config_path` não tem `"base"` no nome — confira que está `acestep-v15-base`.